In [1]:
import numpy as np
import numpy as np
import torch
import datasets
from datasets import Dataset, DatasetDict, Features, Sequence, Value


/share/thickstun/zhihan/.conda/esolm/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
arrs_train = np.load("datasets/sudoku-train-data-001.npy")
arrs_val = np.load("datasets/sudoku-test-data.npy")

In [4]:
def verify_sudoku(g):
    g = np.asarray(g)
    s = set(range(1, 10))
    ok = all(set(g[i, :]) == s for i in range(9)) and all(set(g[:, j]) == s for j in range(9))
    ok = ok and all(set(g[r:r+3, c:c+3].ravel()) == s for r in (0,3,6) for c in (0,3,6))
    return ok


In [5]:
def prepare_sudoku_puzzle_batch(arr):
    B = arr.shape[0]

    t = arr[:, 1:].reshape(B, -1, 4)
    rows = t[:, :, 0]
    cols = t[:, :, 1]
    vals = t[:, :, 2]

    grid = np.zeros((B, 9, 9), dtype=int)
    batch_idx = np.arange(B)[:, None]

    grid[batch_idx, rows, cols] = vals
    return grid.reshape(B, 81)

In [6]:
def prepare_sudoku_dataset(arrs):
    out = np.memmap(
        "sudoku_grids.uint8",
        dtype=np.uint8,
        mode="w+",
        shape=(arrs.shape[0], 81),
    )

    for start in range(0, arrs.shape[0], 1000):
        end = min(start + 1000, arrs.shape[0])

        batch = arrs[start:end]
        grids = prepare_sudoku_puzzle_batch(batch)

        out[start:end] = grids.astype(np.uint8)
    return out

In [8]:
sudoku_train = prepare_sudoku_dataset(arrs_train)
sudoku_train = torch.from_numpy(sudoku_train).long()

In [32]:
sudoku_val = prepare_sudoku_dataset(arrs_val)
sudoku_val = torch.from_numpy(sudoku_val).long()

In [33]:
# sudoku_train = prepare_sudoku_dataset(arrs_train)
# sudoku_val = prepare_sudoku_dataset(arrs_val)

features = Features({
    "input_ids": Sequence(Value("uint8"), length=81),
    "attention_mask": Sequence(Value("uint8"), length=81),
})

# ds_train = Dataset.from_dict(
#     {
#         "input_ids": sudoku_train,                                # (N,81) numpy array
#         "attention_mask": torch.ones_like(sudoku_train), # (N,81)
#     },
#     features=features,
# )

ds_val = Dataset.from_dict(
    {
        "input_ids": sudoku_val,                                # (N,81) numpy array
        "attention_mask": np.ones_like(sudoku_val, np.uint8), # (N,81)
    },
    features=features,
)

In [34]:
ds_val = ds_val.with_format('torch')

In [35]:
dsd = DatasetDict({"train": ds_train, "validation": ds_val})
dsd.push_to_hub('zhihanyang/sudoku')

Creating parquet from Arrow format: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1805/1805 [00:05<00:00, 354.86ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
Upload 0 LFS files: 0it [00:00, ?it/s]
Creating parquet from Arrow format: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 317.86ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
Uploading the dataset shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.26it/s]


CommitInfo(commit_url='https://huggingface.co/datasets/zhihanyang/sudoku/commit/6a52ce7dcafa3a7925cd2ae2252adfbc25ff08aa', commit_message='Upload dataset', commit_description='', oid='6a52ce7dcafa3a7925cd2ae2252adfbc25ff08aa', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/zhihanyang/sudoku', endpoint='https://huggingface.co', repo_type='dataset', repo_id='zhihanyang/sudoku'), pr_revision=None, pr_num=None)

In [ ]:
ds = datasets.load_dataset("zhihanyang/sudoku", split=)

TypeError: must be called with a dataclass type or instance

In [23]:
from datasets import load_dataset

ds = load_dataset("zhihanyang/gsm8k-abel-7b-001")

Generating test split: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 374/374 [00:00<00:00, 49245.61 examples/s]


In [ ]:
def prepare_sudoku_puzzle(arr):
    t = arr[1:].reshape(-1, 4)
    rows = t[:, 0]
    cols = t[:, 1]
    vals = t[:, 2]
    grid = np.zeros((9, 9), dtype=int)
    grid[rows, cols] = vals          # place correct values
    return grid

def verify_sudoku(g):
    g = np.asarray(g)
    s = set(range(1, 10))
    ok = all(set(g[i, :]) == s for i in range(9)) and all(set(g[:, j]) == s for j in range(9))
    ok = ok and all(set(g[r:r+3, c:c+3].ravel()) == s for r in (0,3,6) for c in (0,3,6))
    return ok
